# Gold App — painel do gestor

Camada Gold do gestor, com grão separado por tipo de uso:

- `gold_app_gestor_kpis`: uma linha por CD, pois cada gestor acompanha o CD ao qual está associado.
- `gold_app_gestor_prioridades`: uma linha por alerta aberto no CD, com temperatura, gravidade, prazo e responsável.
- `gold_app_gestor_tendencia`: leituras por câmara para o gráfico de 6 horas, 24 horas ou 7 dias.

## Regras consideradas

- Câmaras estáveis, fora da faixa, offline e sem dados vêm do estado térmico/conectividade da câmara; são independentes do status do alerta.
- Alertas abertos são aqueles com status `ATIVO` ou `RECONHECIDO`. Alertas críticos, sem responsável, em atendimento e SLA vencido contam apenas alertas abertos.
- O limite offline é 10 minutos, conforme dois ciclos de coleta de 5 minutos.
- Regra provisória de SLA: escalonamento em 40% e 70%; próximo do vencimento a partir de 70%; vencido em 100% da vida útil de referência, a partir do horário do alerta. Rever quando o grupo confirmar.
- Prioridade ordena primeiro gravidade (crítico, atenção, baixo); em empate, SLA vencido, próximo do vencimento, em atendimento e pendente; por fim, alertas mais antigos.

O filtro de `id_cd` deve ser aplicado no dashboard para o CD do gestor. As tabelas abaixo guardam dados por CD; o filtro interativo do dashboard não substitui controle de acesso.


In [ ]:
import re

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

dbutils.widgets.text("catalogo_destino", "workspace")
dbutils.widgets.text("schema_silver", "silver_app")
dbutils.widgets.text("schema_gold", "gold_app")
dbutils.widgets.dropdown(
    "tabela_amostra", "gold_app_gestor_kpis",
    ["gold_app_gestor_kpis", "gold_app_gestor_prioridades", "gold_app_gestor_tendencia"],
)
CATALOGO = dbutils.widgets.get("catalogo_destino").strip()
SCHEMA_SILVER = dbutils.widgets.get("schema_silver").strip()
SCHEMA_GOLD = dbutils.widgets.get("schema_gold").strip()
TABELA_AMOSTRA = dbutils.widgets.get("tabela_amostra").strip()

# Hipótese provisória de prazo, sujeita à confirmação do grupo.
PCT_ESCALONAMENTO_GESTOR = 0.40
PCT_ESCALONAMENTO_ADMIN = 0.70
PCT_PROXIMO_VENCIMENTO = PCT_ESCALONAMENTO_ADMIN
PCT_SLA_VENCIDO = 1.00
MINUTOS_OFFLINE = 10


def validar_identificador(valor: str, nome: str) -> str:
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", valor):
        raise ValueError(f"Identificador inválido em {nome}: {valor!r}")
    return valor


CATALOGO = validar_identificador(CATALOGO, "catalogo_destino")
SCHEMA_SILVER = validar_identificador(SCHEMA_SILVER, "schema_silver")
SCHEMA_GOLD = validar_identificador(SCHEMA_GOLD, "schema_gold")
SILVER = f"{CATALOGO}.{SCHEMA_SILVER}"
GOLD = f"{CATALOGO}.{SCHEMA_GOLD}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")


def ler_silver(nome: str) -> DataFrame:
    tabela = f"{SILVER}.silver_app_{validar_identificador(nome, 'nome_tabela')}"
    if not spark.catalog.tableExists(tabela):
        raise ValueError(f"Tabela Silver não encontrada: {tabela}. Execute a Silver primeiro.")
    return spark.table(tabela)


def gravar_gold(df: DataFrame, nome: str) -> None:
    tabela = f"{GOLD}.{validar_identificador(nome, 'nome_tabela')}"
    (
        df.withColumn("_gold_built_at", F.current_timestamp())
        .write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(tabela)
    )
    print(f"Gravada: {tabela}")


def intervalo_para_segundos(coluna):
    """Converte intervalos JDBC em HH:MM[:SS] ou texto de unidade PostgreSQL."""
    valor = F.lower(F.trim(coluna.cast("string")))
    hhmm = valor.rlike(r"^-?\d+:\d{1,2}(:\d{1,2}(\.\d+)?)?$")
    horas_hhmm = F.regexp_extract(valor, r"^(-?\d+):", 1).cast("double")
    minutos_hhmm = F.regexp_extract(valor, r"^-?\d+:(\d{1,2})", 1).cast("double")
    segundos_hhmm = F.regexp_extract(valor, r"^-?\d+:\d{1,2}:(\d{1,2}(\.\d+)?)$", 1).cast("double")
    horas_txt = F.regexp_extract(valor, r"(-?\d+(?:\.\d+)?)\s*(?:hours?|hrs?)\b", 1).cast("double")
    minutos_txt = F.regexp_extract(valor, r"(-?\d+(?:\.\d+)?)\s*(?:minutes?|mins?)\b", 1).cast("double")
    segundos_txt = F.regexp_extract(valor, r"(-?\d+(?:\.\d+)?)\s*(?:seconds?|secs?)\b", 1).cast("double")
    horas = F.when(hhmm, horas_hhmm).otherwise(F.coalesce(horas_txt, F.lit(0.0)))
    minutos = F.when(hhmm, minutos_hhmm).otherwise(F.coalesce(minutos_txt, F.lit(0.0)))
    segundos = F.when(hhmm, F.coalesce(segundos_hhmm, F.lit(0.0))).otherwise(F.coalesce(segundos_txt, F.lit(0.0)))
    reconhecido = valor.isNotNull() & (hhmm | valor.rlike(r"hours?|hrs?|minutes?|mins?|seconds?|secs?"))
    return F.when(reconhecido, horas * 3600 + minutos * 60 + segundos)


cd = ler_silver("cd")
usuario = ler_silver("usuario")
camara = ler_silver("estado_camara_atual")
alerta = ler_silver("alerta")
atendimento = ler_silver("atendimento")
leitura = ler_silver("leitura_temperatura")


In [ ]:
# Uma linha atual de atendimento por alerta.
janela_atendimento = Window.partitionBy("id_alerta").orderBy(
    F.col("data_hora_reconhecimento").desc_nulls_last(), F.col("id_atendimento").desc()
)
atendimento_atual = (
    atendimento.withColumn("_ordem", F.row_number().over(janela_atendimento))
    .filter(F.col("_ordem") == 1).drop("_ordem")
)

# Câmaras: estado da temperatura é independente do estado operacional do alerta.
camaras = (
    camara.alias("c")
    .join(cd.select("id_cd", "nome_cd").alias("d"), "id_cd", "left")
    .select(
        F.col("c.id_camara"), F.col("c.id_cd"), F.col("d.nome_cd"),
        F.col("c.modelo_camara"), F.col("c.localizacao"),
        F.col("c.id_termometro"), F.col("c.temperatura_atual"),
        F.col("c.temperatura_min"), F.col("c.temperatura_max"),
        F.col("c.ultima_leitura_em"), F.col("c.estado_conectividade"),
        F.col("c.temperatura_fora_da_faixa"), F.col("c.estado_termico"),
    )
)

# Converte possíveis acentos e variações de gênero mantendo o rótulo de origem.
chave_gravidade = F.translate(
    F.upper(F.trim(F.col("nivel_gravidade"))),
    "ÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ", "AAAAAEEEEIIIIOOOOOUUUUC",
)
alertas = (
    alerta.alias("a")
    .join(atendimento_atual.alias("t"), "id_alerta", "left")
    .join(camaras.alias("c"), F.col("a.id_camara") == F.col("c.id_camara"), "left")
    .join(
        usuario.select(
            F.col("id_usuario").alias("_id_usuario"),
            F.col("nome_usuario").alias("nome_responsavel"),
        ).alias("u"),
        F.col("t.id_usuario_responsavel") == F.col("u._id_usuario"), "left",
    )
    .select(
        F.col("a.id_alerta"), F.col("a.id_camara"), F.col("c.id_cd"), F.col("c.nome_cd"),
        F.col("c.modelo_camara"), F.col("c.localizacao"),
        F.col("c.temperatura_atual"), F.col("c.temperatura_min"), F.col("c.temperatura_max"),
        F.col("c.temperatura_fora_da_faixa"), F.col("c.estado_termico"), F.col("c.estado_conectividade"),
        F.col("a.data_hora_alerta"), F.col("a.vida_util_referencia_horas"),
        F.col("a.nivel_atual"), F.col("a.nivel_gravidade"), F.col("a.status_alerta"),
        F.col("t.id_atendimento"), F.col("t.id_usuario_responsavel"), F.col("u.nome_responsavel"),
        F.col("t.status_atendimento"), F.col("t.data_hora_reconhecimento"), F.col("t.data_hora_resolucao"),
    )
    .withColumn("_gravidade_chave", chave_gravidade)
    .withColumn(
        "gravidade_padronizada",
        F.when(F.col("_gravidade_chave").isin("BAIXO", "BAIXA"), "BAIXO")
        .when(F.col("_gravidade_chave") == "ATENCAO", "ATENCAO")
        .when(F.col("_gravidade_chave").isin("CRITICO", "CRITICA"), "CRITICO")
        .otherwise("OUTRA"),
    )
    .drop("_gravidade_chave")
)

fim_contagem = F.coalesce(F.col("data_hora_resolucao"), F.current_timestamp())
alertas = (
    alertas
    .withColumn("vida_util_referencia_segundos", intervalo_para_segundos(F.col("vida_util_referencia_horas")))
    .withColumn("tempo_decorrido_segundos", F.greatest(
        F.lit(0), F.unix_timestamp(fim_contagem) - F.unix_timestamp("data_hora_alerta")
    ))
    .withColumn(
        "percentual_consumido_sla",
        F.when(F.col("vida_util_referencia_segundos") > 0,
               F.round(F.col("tempo_decorrido_segundos") / F.col("vida_util_referencia_segundos"), 4)),
    )
    .withColumn(
        "estado_sla",
        F.when(F.col("vida_util_referencia_segundos").isNull() | (F.col("vida_util_referencia_segundos") <= 0), "SEM_REFERENCIA")
        .when(F.col("percentual_consumido_sla") >= F.lit(PCT_SLA_VENCIDO), "SLA_VENCIDO")
        .when(F.col("percentual_consumido_sla") >= F.lit(PCT_PROXIMO_VENCIMENTO), "PROXIMO_VENCIMENTO")
        .otherwise("NO_PRAZO"),
    )
    .withColumn("em_aberto", F.col("status_alerta").isin("ATIVO", "RECONHECIDO"))
    .withColumn("resolvido_hoje", (F.col("status_alerta") == "RESOLVIDO") & (F.to_date("data_hora_resolucao") == F.current_date()))
    .withColumn("tempo_desde_alerta_minutos", F.when(
        F.col("em_aberto"), F.greatest(F.lit(0), F.ceil((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp("data_hora_alerta")) / 60))
    ))
    .withColumn(
        "prioridade_gravidade",
        F.when(F.col("gravidade_padronizada") == "CRITICO", 1)
        .when(F.col("gravidade_padronizada") == "ATENCAO", 2)
        .when(F.col("gravidade_padronizada") == "BAIXO", 3)
        .otherwise(4),
    )
    .withColumn(
        "prioridade_sla",
        F.when(F.col("estado_sla") == "SLA_VENCIDO", 1)
        .when(F.col("estado_sla") == "PROXIMO_VENCIMENTO", 2)
        .when(F.col("estado_sla") == "NO_PRAZO", 3)
        .otherwise(4),
    )
    .withColumn(
        "prioridade_atendimento",
        F.when(F.col("status_atendimento") == "EM_ANDAMENTO", 1)
        .when(F.col("status_atendimento") == "PENDENTE", 2)
        .otherwise(3),
    )
)


In [ ]:
# Indicadores térmicos: contagem pelo estado mais recente de cada câmara.
kpis_camara = (
    camaras.groupBy("id_cd", "nome_cd").agg(
        F.countDistinct("id_camara").alias("total_camaras"),
        F.countDistinct(F.when(F.col("estado_termico") == "NORMAL", F.col("id_camara"))).alias("camaras_estaveis"),
        F.countDistinct(F.when(F.col("estado_termico") == "FORA_DA_FAIXA", F.col("id_camara"))).alias("camaras_em_alerta_temperatura"),
        F.countDistinct(F.when(F.col("estado_conectividade") == "OFFLINE", F.col("id_camara"))).alias("camaras_offline"),
        F.countDistinct(F.when(F.col("estado_conectividade") == "SEM_DADOS", F.col("id_camara"))).alias("camaras_sem_dados"),
    )
)

# Indicadores de alertas: apenas alertas abertos entram nos cartões operacionais.
kpis_alerta = (
    alertas.groupBy("id_cd").agg(
        F.sum(F.when(F.col("em_aberto") & (F.col("gravidade_padronizada") == "CRITICO"), 1).otherwise(0)).alias("alertas_criticos"),
        F.sum(F.when(F.col("em_aberto") & F.col("id_usuario_responsavel").isNull(), 1).otherwise(0)).alias("alertas_sem_responsavel"),
        F.sum(F.when(F.col("em_aberto") & (F.col("status_atendimento") == "EM_ANDAMENTO"), 1).otherwise(0)).alias("alertas_em_atendimento"),
        F.sum(F.when(F.col("em_aberto") & (F.col("estado_sla") == "SLA_VENCIDO"), 1).otherwise(0)).alias("sla_vencido"),
        F.sum(F.when(F.col("resolvido_hoje"), 1).otherwise(0)).alias("resolvidos_hoje"),
    )
)

kpis = (
    kpis_camara.join(kpis_alerta, "id_cd", "left")
    .fillna(0, subset=[
        "alertas_criticos", "alertas_sem_responsavel", "alertas_em_atendimento",
        "sla_vencido", "resolvidos_hoje",
    ])
)

prioridades = (
    alertas.filter(F.col("em_aberto"))
    .withColumn(
        "prioridade_status", F.col("prioridade_atendimento")
    )
    .drop("em_aberto", "resolvido_hoje", "vida_util_referencia_segundos", "tempo_decorrido_segundos", "prioridade_atendimento")
)

# Série histórica de todas as leituras por câmara; filtros de data e alerta são aplicados no dashboard.
tendencia = (
    leitura.alias("l")
    .join(
        camaras.select(
            "id_termometro", "id_camara", "id_cd", "nome_cd", "localizacao",
            "temperatura_min", "temperatura_max",
        ).alias("c"),
        F.col("l.id_termometro") == F.col("c.id_termometro"), "inner",
    )
    .select(
        F.col("c.id_cd"), F.col("c.nome_cd"), F.col("c.id_camara"), F.col("c.localizacao"),
        F.col("l.id_alerta"), F.col("l.id_leitura"), F.col("l.data_hora_leitura"),
        F.col("l.temperatura"), F.col("c.temperatura_min"), F.col("c.temperatura_max"),
        ((F.col("l.temperatura") < F.col("c.temperatura_min"))
         | (F.col("l.temperatura") > F.col("c.temperatura_max"))).alias("temperatura_fora_da_faixa"),
    )
)


In [ ]:
TABELAS_GOLD = {
    "gold_app_gestor_kpis": kpis,
    "gold_app_gestor_prioridades": prioridades,
    "gold_app_gestor_tendencia": tendencia,
}
for nome, df in TABELAS_GOLD.items():
    gravar_gold(df, nome)
print(f"Gold do gestor concluída: {len(TABELAS_GOLD)} tabelas em {GOLD}")


In [ ]:
TABELAS_AMOSTRA = set(TABELAS_GOLD)
if TABELA_AMOSTRA not in TABELAS_AMOSTRA:
    raise ValueError(f"tabela_amostra inválida: {TABELA_AMOSTRA}")
tabela = f"{GOLD}.{TABELA_AMOSTRA}"
df_amostra = spark.table(tabela)
print(f"Amostra: {tabela}")
df_amostra.printSchema()
display(df_amostra.limit(20))
